In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [ ]:
torch.manual_seed(42)

In [ ]:
#I loaded this with AI
import torchvision
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

# Download and load the training data
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)

# Download and load the test data
test_data = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)

cpu


In [ ]:
# Extract images and labels from the PyTorch dataset
# Flatten the 28x28 images into 784 features
X_raw = train_data.data.numpy().reshape(-1, 28*28)
y_raw = train_data.targets.numpy()
X_test_raw = test_data.data.numpy().reshape(-1, 28*28)
y_test_raw = test_data.targets.numpy()

data = pd.DataFrame(X_test_raw)
data.insert(0, 'label', y_test_raw)

# Create a DataFrame
df = pd.DataFrame(X_raw)

# Insert the label as the first column to match your previous setup
df.insert(0, 'label', y_raw)
df


,label,0,1,2,3,4,5,6,7,8,...,774,775,776,777,778,779,780,781,782,783
0,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,...,119,114,130,76,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,33,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,0,0,0,0,0,0,0,0,0,0,...,66,54,50,5,0,1,0,0,0,0


In [ ]:
X_train = df.iloc[:,1:].values/255.0
y_train = df.iloc[:,0].values

X_test = data.iloc[:,1:].values/255.0
y_test = data.iloc[:,0].values

In [ ]:
class CustomDataset(Dataset):
  def __init__(self, feature, label):
    self.feature = torch.tensor(feature, dtype = torch.float32).reshape(-1,1,28,28)
    self.label = torch.tensor(label, dtype = torch.long)

  def __len__(self):
    return len(self.feature)
  def __getitem__(self,idx):
    return self.feature[idx], self.label[idx]

In [ ]:
train_data = CustomDataset(X_train,y_train)
test_data = CustomDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
test_laoder = DataLoader(test_data, batch_size =32, shuffle = False)

In [ ]:
class MyNN(nn.Module):

  def __init__(self,input_feature):
    super().__init__()
    self.feature = nn.Sequential(
    nn.Conv2d(input_feature, 32, kernel_size = 3, padding= 'same'),
    nn.ReLU(),
    nn.BatchNorm2d(32),
    nn.MaxPool2d(kernel_size = 2, stride = 2 ),

    nn.Conv2d(32, 64, kernel_size = 3, padding= 'same'),
    nn.ReLU(),
    nn.BatchNorm2d(64),
    nn.MaxPool2d(kernel_size = 2, stride = 2),

    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p = 0.4),
        nn.Linear(64,10)
    )
  def forward(self,x):
    x = self.feature(x)
    x = self.classifier(x)

    return x

In [ ]:
learning_rate = 0.01
epochs = 100

In [ ]:
model = MyNN(1)

model.to(device)
criterian = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr = learning_rate, weight_decay = 1e-4)

In [ ]:
#training loop

for epoch in range(epochs):

  total_epoch_loss = 0
  for batch_feature, batch_label in train_loader:
    # Move data to GPU if available
    batch_feature = batch_feature.to(device)
    batch_label = batch_label.to(device)

    #forward
    output = model(batch_feature)
    #loss calculation
    loss = criterian(output,batch_label)
    total_epoch_loss = total_epoch_loss + loss.item()
    #backpropagation
    optimizer.zero_grad()
    loss.backward()
    # update
    optimizer.step()

  avg = total_epoch_loss/len(train_loader)
  print(f"Epoch  : {epoch +1}/{epochs}\tLoss  : {avg}")


Epoch  : 1/100	Loss  : 0.5971507949153583
Epoch  : 2/100	Loss  : 0.356778207463026
Epoch  : 3/100	Loss  : 0.30547251666585606
Epoch  : 4/100	Loss  : 0.2746960279926658
Epoch  : 5/100	Loss  : 0.2507318696359793
Epoch  : 6/100	Loss  : 0.23341130875547728
Epoch  : 7/100	Loss  : 0.21959965329368908
Epoch  : 8/100	Loss  : 0.20167907067288954
Epoch  : 9/100	Loss  : 0.1924818929042667
Epoch  : 10/100	Loss  : 0.1801458300665021
Epoch  : 11/100	Loss  : 0.17133656728242835
Epoch  : 12/100	Loss  : 0.16238086390470466
Epoch  : 13/100	Loss  : 0.15427560855249564
Epoch  : 14/100	Loss  : 0.1470079405243198
Epoch  : 15/100	Loss  : 0.13756654032741983
Epoch  : 16/100	Loss  : 0.13413335169088095
Epoch  : 17/100	Loss  : 0.12489975770004094
Epoch  : 18/100	Loss  : 0.1226316364424924
Epoch  : 19/100	Loss  : 0.11199669711515307
Epoch  : 20/100	Loss  : 0.11058705954253674
Epoch  : 21/100	Loss  : 0.1036209507673358
Epoch  : 22/100	Loss  : 0.09841906640011196
Epoch  : 23/100	Loss  : 0.09336134689409907
Epoch  

In [ ]:
model.eval()

MyNN(
  (feature): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [ ]:
#evaluation code
total = 0
correct = 0
with torch.no_grad():
  for epoch in range(epochs):

    for batch_feature, batch_label in test_laoder:
      # Move data to GPU if available
      batch_feature = batch_feature.to(device)
      batch_label = batch_label.to(device)

      output = model(batch_feature)

      _, predicted = torch.max(output,1)
      total = total + batch_label.shape[0]
      correct = correct +(predicted == batch_label).sum().item()

print(correct/total)

0.9233
